# GenAI-Traces: Google Gemini Tracing Test

This notebook tests the GenAI-Traces SDK with Google Gemini.

In [2]:
import sys
sys.path.insert(0, '..')

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../.env')

# Get Gemini API key
GEMINI_API_KEY = os.getenv('gemini', '').strip().strip('"').split('=')[-1].strip()
if not GEMINI_API_KEY:
    # Try alternate format
    for line in open('../.env').readlines():
        if 'gemini' in line.lower():
            GEMINI_API_KEY = line.split('=')[-1].strip()
            break

print(f"Gemini API Key loaded: {'Yes' if GEMINI_API_KEY else 'No'}")
print(f"Key preview: {GEMINI_API_KEY[:10]}..." if GEMINI_API_KEY else "No key found")

Gemini API Key loaded: Yes
Key preview: AIzaSyBh9I...


## 1. Initialize GenAI-Traces

In [3]:
from genai_traces import init_tracer, get_tracer
from genai_traces.config import TracerConfig
from genai_traces.exporters import ConsoleExporter, JSONFileExporter

# Initialize tracer with config
config = TracerConfig(
    service_name="gemini-test",
    environment="development",
    enable_pii_detection=True,
    enable_cost_tracking=True,
)

# Create exporters
console_exporter = ConsoleExporter()
json_exporter = JSONFileExporter(output_dir="../traces", rotation="daily")

# Initialize tracer
tracer = init_tracer(config, exporters=[console_exporter, json_exporter])
print("Tracer initialized!")
print(f"Service: {config.service_name}")

Tracer initialized!
Service: gemini-test


## 2. Setup Gemini Client

In [6]:
try:
    import google.generativeai as genai
    
    # Configure Gemini
    genai.configure(api_key=GEMINI_API_KEY)
    
    # Create model - use gemini-2.0-flash-001 (gemini-2.0-flash-001 is deprecated)
    model = genai.GenerativeModel('gemini-2.0-flash-001')
    print("Gemini client created successfully!")
    print("Model: gemini-2.0-flash-001")
    
except ImportError:
    print("google-generativeai not installed. Installing...")
    !pip install google-generativeai
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-2.5-flash')
    print("Gemini client created!")

Gemini client created successfully!
Model: gemini-2.0-flash-001


## 3. Test Basic Tracing with Decorators

In [7]:
from genai_traces import trace_llm
from genai_traces.core.types import SpanType

@trace_llm(model="gemini-2.0-flash-001", provider="google")
def chat_with_gemini(prompt: str) -> str:
    """Make a request to Gemini."""
    response = model.generate_content(prompt)
    return response.text

# Test the decorated function
print("Testing basic Gemini tracing...")
result = chat_with_gemini("What is the capital of Japan? Answer in one sentence.")
print(f"\nResponse: {result}")

Testing basic Gemini tracing...
[SPAN] chat_with_gemini (llm) - error
{
  "trace_id": "31886c4f5fef464c9c628cbd291c0563",
  "span_id": "5a12c1e31ff5543c",
  "parent_span_id": null,
  "root_span_id": "5a12c1e31ff5543c",
  "name": "chat_with_gemini",
  "span_type": "llm",
  "start_time": "2026-04-04T19:57:51.558340",
  "end_time": "2026-04-04T19:57:51.797840",
  "duration_ms": 239.5,
  "status": "error",
  "status_message": "429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapi

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 8.849063374s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quot

## 4. Test Context Manager Tracing

In [8]:
from genai_traces.core.context_manager import trace_llm_context
import time

print("Testing context manager tracing with Gemini...")

with trace_llm_context(name="gemini_chat_context", model="gemini-2.0-flash-001") as span:
    # Set custom attributes
    span.set_attribute("custom.test", "gemini_context_test")
    span.set_attribute("llm.provider", "google")
    
    start_time = time.time()
    
    # Make the API call
    response = model.generate_content("What is 10 + 20? Just give the number.")
    
    duration = (time.time() - start_time) * 1000
    
    # Record response details
    content = response.text
    span.set_attribute("llm.completion", content)
    span.set_attribute("llm.duration_ms", duration)
    
    print(f"Response: {content}")
    print(f"Duration: {duration:.2f}ms")

Testing context manager tracing with Gemini...
[SPAN] gemini_chat_context (llm) - error
{
  "trace_id": "9c58682bd5b845259bd868a5dc4bb9d2",
  "span_id": "5a12d6cc92f799d6",
  "parent_span_id": null,
  "root_span_id": "5a12d6cc92f799d6",
  "name": "gemini_chat_context",
  "span_type": "llm",
  "start_time": "2026-04-04T19:57:56.911986",
  "end_time": "2026-04-04T19:57:57.067400",
  "duration_ms": 155.414,
  "status": "error",
  "status_message": "429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: gener

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 3.603594681s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quot

## 5. Test Multi-turn Conversation

In [ ]:
from genai_traces.intelligence.conversation import set_conversation_context, Session
from genai_traces import trace_llm

# Create a conversation
context = set_conversation_context(
    conversation_id="gemini-conv-001",
    user_id="test-user",
)

# Start a chat session
chat = model.start_chat(history=[])

@trace_llm(model="gemini-2.0-flash-001", provider="google")
def chat_turn(message: str) -> str:
    response = chat.send_message(message)
    return response.text

print("Testing multi-turn conversation...")

# Turn 1
print("\nUser: My name is Alice.")
response1 = chat_turn("My name is Alice.")
print(f"Gemini: {response1}")

# Turn 2
print("\nUser: What's my name?")
response2 = chat_turn("What's my name?")
print(f"Gemini: {response2}")

# Turn 3
print("\nUser: Tell me a fun fact about the letter A.")
response3 = chat_turn("Tell me a fun fact about the letter A.")
print(f"Gemini: {response3}")

## 6. Test Token Estimation for Gemini

In [ ]:
from genai_traces.telemetry.tokens.estimator import TokenEstimator
from genai_traces.telemetry.cost.estimator import CostEstimator

estimator = TokenEstimator()
cost_estimator = CostEstimator()

test_prompt = "Explain the theory of relativity in simple terms for a 10-year-old."

# Estimate tokens (using GPT tokenizer as approximation)
estimated_tokens = estimator.estimate_prompt_tokens(test_prompt, model="gpt-4")
print(f"Estimated prompt tokens: {estimated_tokens}")

# Make actual call and compare
with trace_llm_context(name="gemini_token_test", model="gemini-2.0-flash-001") as span:
    response = model.generate_content(test_prompt)
    content = response.text
    
    # Estimate response tokens
    response_tokens = estimator.estimate_prompt_tokens(content, model="gpt-4")
    
    span.set_attribute("llm.prompt.tokens.estimated", estimated_tokens)
    span.set_attribute("llm.completion.tokens.estimated", response_tokens)
    
    print(f"Estimated response tokens: {response_tokens}")
    print(f"Total estimated tokens: {estimated_tokens + response_tokens}")
    print(f"\nResponse preview: {content[:200]}...")

## 7. Test Evaluation

In [ ]:
from genai_traces.intelligence.evaluation import RelevanceEvaluator
from genai_traces.intelligence.evaluation.coherence import CoherenceEvaluator

# Create evaluators
relevance_eval = RelevanceEvaluator()
coherence_eval = CoherenceEvaluator()

# Test data
test_prompt = "What are the benefits of exercise?"
test_response = "Regular exercise improves cardiovascular health, boosts mood, helps maintain healthy weight, and increases energy levels."

print("Evaluating Gemini response quality...")
print(f"\nPrompt: {test_prompt}")
print(f"Response: {test_response}")

# Evaluate relevance
relevance_result = relevance_eval.evaluate(
    prompt=test_prompt,
    response=test_response,
)
print(f"\nRelevance Score: {relevance_result['score']:.2f}")

# Evaluate coherence
coherence_result = coherence_eval.evaluate(
    prompt=test_prompt,
    response=test_response,
)
print(f"Coherence Score: {coherence_result['score']:.2f}")

## 8. Test Anomaly Detection

In [ ]:
from genai_traces.telemetry.anomaly import AnomalyDetector
from genai_traces.telemetry.anomaly.baselines import ModelBaseline
import random

# Create baseline tracker
baseline = ModelBaseline(window_size=100)

# Simulate latency data
print("Simulating latency data for anomaly detection...")

# Normal latencies (100-300ms)
for _ in range(50):
    latency = random.gauss(200, 30)
    baseline.add("gemini-2.0-flash-001", "latency", latency)

# Get baseline stats
stats = baseline.get_stats("gemini-2.0-flash-001", "latency")
print(f"\nBaseline Statistics:")
print(f"  Mean: {stats.mean:.2f}ms")
print(f"  Std: {stats.std:.2f}ms")
print(f"  Min: {stats.min_value:.2f}ms")
print(f"  Max: {stats.max_value:.2f}ms")

# Test anomaly detection
test_values = [200, 250, 180, 800, 150, 1200]  # 800 and 1200 are anomalies
print(f"\nTesting anomaly detection:")
for val in test_values:
    is_anomaly = baseline.is_anomaly("gemini-2.0-flash-001", "latency", val)
    status = "ANOMALY" if is_anomaly else "NORMAL"
    print(f"  {val}ms -> {status}")

## 9. Test Prompt Registry

In [1]:
from genai_traces.prompt_management import PromptRegistry

# Create prompt registry
registry = PromptRegistry()

# Register prompts
registry.register(
    name="greeting",
    template="Hello {name}! How can I help you today?",
    version="1.0.0",
    description="Standard greeting prompt",
)

registry.register(
    name="summarize",
    template="Please summarize the following text in {num_sentences} sentences:\n\n{text}",
    version="1.0.0",
    description="Text summarization prompt",
)

# Render prompts
greeting = registry.render("greeting", name="Alice")
print(f"Greeting prompt: {greeting}")

summary_prompt = registry.render(
    "summarize",
    num_sentences=2,
    text="Artificial intelligence is transforming many industries. It enables automation of complex tasks and provides insights from large datasets."
)
print(f"\nSummary prompt: {summary_prompt}")

# Use with Gemini
with trace_llm_context(name="prompt_registry_test", model="gemini-2.0-flash-001") as span:
    span.set_attribute("prompt.name", "summarize")
    span.set_attribute("prompt.version", "1.0.0")
    
    response = model.generate_content(summary_prompt)
    print(f"\nGemini response: {response.text}")

ModuleNotFoundError: No module named 'genai_traces'

## 10. View Exported Traces

In [ ]:
import json
from pathlib import Path

# Flush all exporters
tracer.flush()

# Read and display traces
trace_file = Path("../traces/gemini_traces.jsonl")
if trace_file.exists():
    print("Exported Gemini Traces:")
    print("=" * 60)
    with open(trace_file) as f:
        lines = f.readlines()
        print(f"Total traces: {len(lines)}")
        for i, line in enumerate(lines[-5:], 1):  # Show last 5 traces
            trace = json.loads(line)
            print(f"\nTrace {i}:")
            print(f"  Name: {trace.get('name', 'N/A')}")
            print(f"  Type: {trace.get('span_type', 'N/A')}")
            print(f"  Duration: {trace.get('duration_ms', 'N/A')}ms")
            print(f"  Status: {trace.get('status', 'N/A')}")
else:
    print("No trace file found yet.")

## Summary

This notebook tested:
1. Tracer initialization for Gemini
2. Gemini client setup
3. Decorator-based tracing
4. Context manager tracing
5. Multi-turn conversation tracking
6. Token estimation
7. Response evaluation
8. Anomaly detection
9. Prompt registry
10. Trace export and viewing